<div style="background-color: #f9f9a9; padding: 1px; border-radius: 1px;color:blue;text-align:center;">
  <h2>드론 카메라 활용 : 영상녹화</h2>
</div>

In [ ]:
import time
import cv2
from threading import Thread
from djitellopy import Tello

# ── 연결 및 스트림 시작 ────────────────────────────
tello = Tello(host='192.168.10.1')
tello.connect()
print(f"[연결] 배터리: {tello.get_battery()}%")

tello.streamon()
time.sleep(2)                          # ✅ 스트림 안정화 대기

frame_read = tello.get_frame_read()
keepRecording = True


# ── 녹화 함수 ──────────────────────────────────────
def videoRecorder():

    # ✅ 유효한 프레임이 올 때까지 대기
    print("[녹화] 프레임 대기 중...")
    timeout = 0
    while frame_read.frame is None:
        time.sleep(0.1)
        timeout += 1
        if timeout > 50:               # 5초 초과 시 중단
            print("[오류] 프레임 수신 실패 - 녹화 중단")
            return

    time.sleep(1)                      # ✅ 첫 프레임 안정화 추가 대기

    # ✅ 프레임 크기 확인 후 VideoWriter 초기화
    frame = frame_read.frame
    height, width, _ = frame.shape
    print(f"[녹화] 해상도: {width} x {height}")

    video = cv2.VideoWriter(
        'video.avi',
        cv2.VideoWriter_fourcc(*'XVID'),
        30,
        (width, height)
    )

    # ✅ VideoWriter 정상 초기화 여부 확인
    if not video.isOpened():
        print("[오류] VideoWriter 초기화 실패 - 코덱 확인 필요")
        return

    print("[녹화] 녹화 시작!")
    frame_count = 0

    while keepRecording:
        current_frame = frame_read.frame
        if current_frame is not None:  # ✅ None 체크 후 저장
            video.write(current_frame)
            frame_count += 1
        time.sleep(1 / 30)

    video.release()                    # ✅ 파일 정상 저장
    print(f"[녹화] 완료 - 총 {frame_count}프레임 / "
          f"약 {frame_count // 30}초 저장")


# ── 녹화 스레드 시작 ───────────────────────────────
recorder = Thread(target=videoRecorder)
recorder.start()
time.sleep(2)                          # ✅ 녹화 준비 후 비행 시작


# ── 드론 비행 ──────────────────────────────────────
tello.takeoff()
tello.move_up(70)
tello.rotate_counter_clockwise(360)
tello.land()


# ── ✅ 최소 10초 녹화 보장 ─────────────────────────
print("[대기] 10초 추가 녹화 중...")
time.sleep(10)


# ── 녹화 종료 ──────────────────────────────────────
keepRecording = False
recorder.join()

tello.streamoff()
tello.end()
print("[완료] 영상 저장 완료: video.avi")

In [ ]:
#현재 열린 UDP 포트 확인
!netstat -anop udp

<div style="background-color: #f9f9a9; padding: 1px; border-radius: 15px;color:blue;text-align:center;">
  <h2>드론 프로그램 종료하기</h2>
</div>

In [ ]:
#udp(8890) 실행후 대기중인 프로세스를 종료
import subprocess
import os
import platform
    
def kill_process_using_port(port):
    if platform.system() == "Windows":
        cmd = f"netstat -ano | findstr :{port}"
    else:
        # lsof 툴 설치: sudo apt install lsof
        # 옵션 -i :port|TCP|UDP : 모든 인터넷연결포트 확인. 그외 옵션 TCP, UDP, 또는 :port 
        # 옵션 -p PID 특정PID 목록보기
        cmd = f"lsof -i :{port}"
    try:
        result = subprocess.check_output(cmd, shell=True, encoding='utf-8')
        #print(f"포트 {port} 사용 프로세스 목록")
        #print(result)
        pid=result.split()
        if platform.system() == "Windows":
            os.system(f"taskkill /PID {pid[3]} /F")
        else:
            os.system(f"kill -9 {pid[3]}")
        print(f"{port} -> {pid[3]} 강제 종료 완료")        
    except subprocess.CalledProcessError:
        print(f"포트 {port}를 사용하는 프로세스를 찾을 수 없음")
        
#프로그램 종료
kill_process_using_port(8890)

## ✈️ djitellopy 기본 비행 명령어

<span style="display:inline-block;position:left;">

| 메서드                                                   | 설명                                  |                                               |
| ----------------------------------------------------- | ----------------------------------- | --------------------------------------------- |
| `connect()`                                           | 드론과 연결                              |                                               |
| `takeoff()`                                           | 이륙                                  |                                               |
| `land()`                                              | 착륙                                  |                                               |
| `emergency()`                                         | 비상 정지 (모터 즉시 정지)                    |                                               |
| `move_up(x)` / `move_down(x)`                         | x(20 ~ 500 cm) 만큼 상승 / 하강                    |                                               |
| `move_left(x)` / `move_right(x)`                      | x(20 ~ 500 cm) 만큼 좌 / 우 이동                   |                                               |
| `move_forward(x)` / `move_back(x)`                    | x(20 ~ 500 cm) 만큼 전진 / 후진                    |                                               |
| `rotate_clockwise(x)` / `rotate_counter_clockwise(x)` | x(1 ~ 360 도) 만큼 시계 / 반시계 방향 회전              |                                               |
| `flip(direction)`                                     | 지정된 방향으로 플립 (예: 'l', 'r', 'f', 'b') |                                               |
| `go_xyz_speed(x, y, z, speed)`                        | 지정된 좌표로 이동 (cm 단위)<br> x,y,z : -500 ~ 500<br>speed: 10 ~ 100  |                                               |
| `curve_xyz_speed(...)`                                | 곡선 경로로 이동<br>x1,y1,z1,x2,y2,z2: -500 ~ 500<br>speed: 10 ~ 60 |                                               |
| `send_rc_control(lr, fb, ud, yaw)`                    | 실시간 RC 제어 (각 방향 속도 설정)              | 조이스틱 활용 |

</span>